# Final Ablation & Statistical Validation — Full Colab

**Project:** Agentic Explainable Agricultural Decision Support System using Multi-Agent AI, RAG and Machine Learning

This is the **final implementation-validation notebook** for the current conference-paper experiment.

It does **not regenerate Qwen answers**. It reuses the saved P2 experiment outputs from:
`MyDrive/Capstone_Conference_Paper/`

## What this notebook does automatically

1. Reconstructs the exact leakage-free train/validation/test split used in P1/P2.
2. Re-trains the confidence estimator under multiple feature ablations.
3. Selects **each ablation's adaptive thresholds using validation only**.
4. Evaluates every ablation on the untouched test set.
5. Measures which signals actually contribute:
   - Original retrieval-rank signal
   - MiniLM semantic signal
   - Cross-Encoder signal
   - Rank-derived signals
6. Computes paired bootstrap **95% confidence intervals**.
7. Runs paired sign-flip permutation tests.
8. Applies Holm correction for multiple comparisons.
9. Statistically compares **Original Retrieval vs Full P2**.
10. Statistically compares **G1 Original RAG vs G3 P2 Adaptive RAG** using the already-generated Qwen outputs.
11. Creates paper-ready CSVs, PNG figures, and an automatic statistical summary.

## Scientific rules

- All thresholds are tuned on **validation only**.
- The test set remains untouched during tuning.
- Query-level pairing is preserved in bootstrap/permutation analysis.
- `Dataset-Response Similarity` remains a **proxy**, not gold-answer accuracy.
- `Semantic Evidence-Support` remains an embedding-based **proxy**, not a true faithfulness metric.
- A difference is not called statistically reliable unless its interval/test supports that interpretation.

### Run
This notebook does **not require a GPU**.  
Open in Colab and click **Runtime → Run all**.


In [ ]:
# 0. CONFIG + IMPORTS + GOOGLE DRIVE
SEED = 42
TOP_K = 5
MIN_COVERAGE = 0.90
N_BOOT = 10000
N_PERM = 20000

DRIVE_DIR = "/content/drive/MyDrive/Capstone_Conference_Paper"

import os, re, random, warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from google.colab import drive

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)

drive.mount("/content/drive", force_remount=False)

SAVE_DIR = Path(DRIVE_DIR)
if not SAVE_DIR.exists():
    raise FileNotFoundError(
        "Capstone_Conference_Paper Drive folder not found. "
        "Run the P1/P2 notebooks first."
    )

print("Using saved experiment folder:", SAVE_DIR)
print("No GPU is required for this notebook.")


## 1. Load scored evidence data and reconstruct the exact split

The same `SEED=42`, group split, feature construction, and train/validation/test proportions used in P1/P2 are reused here.


In [ ]:
SCORED_PATH = SAVE_DIR / "R2_all_scored_query_chunk_pairs.csv"

if not SCORED_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SCORED_PATH}. Run the full P1 notebook first."
    )

eval_df = pd.read_csv(SCORED_PATH)

required = [
    "query", "chunk", "chunk_number", "relevance_label",
    "embedding_similarity", "cross_encoder_score"
]

missing = [c for c in required if c not in eval_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

gate_df = eval_df.copy()

gate_df["original_rank_signal"] = 1.0 / gate_df["chunk_number"]

gate_df["bi_rank"] = (
    gate_df.groupby("query")["embedding_similarity"]
    .rank(method="first", ascending=False)
)

gate_df["cross_rank"] = (
    gate_df.groupby("query")["cross_encoder_score"]
    .rank(method="first", ascending=False)
)

gate_df["bi_rank_signal"] = 1.0 / gate_df["bi_rank"]
gate_df["cross_rank_signal"] = 1.0 / gate_df["cross_rank"]

FULL_FEATURES = [
    "original_rank_signal",
    "embedding_similarity",
    "cross_encoder_score",
    "bi_rank_signal",
    "cross_rank_signal",
]

TARGET = "relevance_label"

gate_df = gate_df.dropna(
    subset=FULL_FEATURES + [TARGET]
).copy()

# EXACT P1/P2 split reconstruction
s1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)

train_idx, temp_idx = next(
    s1.split(gate_df, groups=gate_df["query"])
)

train_df = gate_df.iloc[train_idx].copy()
temp_df = gate_df.iloc[temp_idx].copy()

s2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED
)

val_idx, test_idx = next(
    s2.split(temp_df, groups=temp_df["query"])
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_q = set(train_df["query"])
val_q = set(val_df["query"])
test_q = set(test_df["query"])

assert not (
    train_q & val_q
    or train_q & test_q
    or val_q & test_q
), "Query leakage detected."

print("Train queries:", train_df["query"].nunique())
print("Validation queries:", val_df["query"].nunique())
print("Test queries:", test_df["query"].nunique())
print("Query overlap: 0 / 0 / 0")


## 2. Shared adaptive-gate functions

Every ablation is evaluated fairly:

**train estimator → get validation probabilities → tune P2 low/high/context on validation only → freeze → evaluate test**


In [ ]:
def original_metrics(df):
    rows = []

    for query, group in df.groupby("query"):
        base = (
            group
            .sort_values("chunk_number")
            .head(TOP_K)
        )

        total = group["relevance_label"].sum()

        rows.append({
            "query": query,
            "precision": base["relevance_label"].mean(),
            "recall": (
                base["relevance_label"].sum() / total
                if total > 0 else np.nan
            ),
            "top1": int(base.iloc[0]["relevance_label"]),
            "coverage": 1,
            "abstained": 0,
            "context_count": len(base),
        })

    z = pd.DataFrame(rows)

    return {
        "precision": z["precision"].mean(),
        "recall": z["recall"].mean(),
        "top1": z["top1"].mean(),
        "coverage": 1.0,
        "abstention": 0.0,
        "mean_context": z["context_count"].mean(),
    }, z


def adaptive_select(group, low, high, target_context):
    primary = (
        group[group["evidence_confidence"] >= high]
        .sort_values("evidence_confidence", ascending=False)
        .copy()
    )

    secondary = (
        group[
            (group["evidence_confidence"] >= low)
            & (group["evidence_confidence"] < high)
        ]
        .sort_values("evidence_confidence", ascending=False)
        .copy()
    )

    primary["evidence_tier"] = "PRIMARY"
    secondary["evidence_tier"] = "SECONDARY"

    chosen = primary.head(target_context).copy()

    remaining = max(
        0,
        target_context - len(chosen)
    )

    if remaining > 0:
        chosen = pd.concat(
            [chosen, secondary.head(remaining)],
            ignore_index=False
        )

    return chosen


def adaptive_metrics(df, low, high, target_context):
    rows = []

    for query, group in df.groupby("query"):
        selected = adaptive_select(
            group,
            low,
            high,
            target_context
        )

        covered = int(len(selected) > 0)
        total = group["relevance_label"].sum()

        rows.append({
            "query": query,
            "precision": (
                selected["relevance_label"].mean()
                if covered else np.nan
            ),
            "recall": (
                selected["relevance_label"].sum() / total
                if total > 0 and covered else 0.0
            ),
            "top1": (
                int(selected.iloc[0]["relevance_label"])
                if covered else np.nan
            ),
            "coverage": covered,
            "abstained": 1 - covered,
            "context_count": len(selected),
        })

    z = pd.DataFrame(rows)

    p = z["precision"].mean(skipna=True)
    r = z["recall"].mean()

    f1 = (
        2 * p * r / (p + r)
        if (p + r) > 0 else 0.0
    )

    return {
        "precision": p,
        "recall": r,
        "f1": f1,
        "top1": z["top1"].mean(skipna=True),
        "coverage": z["coverage"].mean(),
        "abstention": z["abstained"].mean(),
        "mean_context": z["context_count"].mean(),
    }, z


ORIGINAL_VAL, _ = original_metrics(val_df)
ORIGINAL_TEST, ORIGINAL_TEST_Q = original_metrics(test_df)

print("Original validation precision:", round(ORIGINAL_VAL["precision"], 4))
print("Original test precision:", round(ORIGINAL_TEST["precision"], 4))


## 3. Define the ablation variants

The full model uses all five evidence signals.

Ablations remove one signal family at a time, plus minimal baselines.


In [ ]:
ABLATIONS = {
    "A0 Full P2": [
        "original_rank_signal",
        "embedding_similarity",
        "cross_encoder_score",
        "bi_rank_signal",
        "cross_rank_signal",
    ],

    "A1 No Original Rank": [
        "embedding_similarity",
        "cross_encoder_score",
        "bi_rank_signal",
        "cross_rank_signal",
    ],

    "A2 No MiniLM": [
        "original_rank_signal",
        "cross_encoder_score",
        "cross_rank_signal",
    ],

    "A3 No CrossEncoder": [
        "original_rank_signal",
        "embedding_similarity",
        "bi_rank_signal",
    ],

    "A4 Original Rank Only": [
        "original_rank_signal",
    ],

    "A5 Semantic Scores Only": [
        "embedding_similarity",
        "cross_encoder_score",
    ],

    "A6 Rank Signals Only": [
        "original_rank_signal",
        "bi_rank_signal",
        "cross_rank_signal",
    ],
}

for name, features in ABLATIONS.items():
    print(name, "=>", features)


## 4. Train, validation-tune, and test every ablation

The same P2 search rule is used for every variant:

- query coverage ≥ 90%
- seek validation precision at least 5 percentage points above original validation retrieval
- among eligible settings, maximize precision–recall F1
- tie-break by precision and Top-1 relevance
- if the precision constraint has no eligible configuration, fall back to the coverage constraint only


In [ ]:
def fit_variant(feature_list):
    model = Pipeline([
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=SEED,
            )
        ),
    ])

    model.fit(
        train_df[feature_list],
        train_df[TARGET].astype(int),
    )

    v = val_df.copy()
    t = test_df.copy()

    v["evidence_confidence"] = model.predict_proba(
        v[feature_list]
    )[:, 1]

    t["evidence_confidence"] = model.predict_proba(
        t[feature_list]
    )[:, 1]

    return model, v, t


def tune_adaptive_policy(validation_df):
    min_precision = ORIGINAL_VAL["precision"] + 0.05

    search_rows = []

    for low in np.arange(0.20, 0.56, 0.05):
        for high in np.arange(0.45, 0.81, 0.05):
            if high <= low:
                continue

            for target_context in [2, 3]:
                metrics, _ = adaptive_metrics(
                    validation_df,
                    float(low),
                    float(high),
                    int(target_context),
                )

                search_rows.append({
                    "low_threshold": round(float(low), 2),
                    "high_threshold": round(float(high), 2),
                    "target_context": int(target_context),
                    **metrics,
                })

    grid = pd.DataFrame(search_rows)

    eligible = grid[
        (grid["coverage"] >= MIN_COVERAGE)
        & (grid["precision"] >= min_precision)
    ].copy()

    used_fallback = False

    if eligible.empty:
        used_fallback = True
        eligible = grid[
            grid["coverage"] >= MIN_COVERAGE
        ].copy()

    best = (
        eligible
        .sort_values(
            ["f1", "precision", "top1"],
            ascending=False
        )
        .iloc[0]
    )

    return best, grid, used_fallback


ablation_rows = []
ablation_query_results = {}
ablation_models = {}
ablation_grids = {}

for name, features in ABLATIONS.items():
    print("\nRunning:", name)

    model, v, t = fit_variant(features)

    best, grid, fallback = tune_adaptive_policy(v)

    low = float(best["low_threshold"])
    high = float(best["high_threshold"])
    target_context = int(best["target_context"])

    test_metrics, test_query_metrics = adaptive_metrics(
        t,
        low,
        high,
        target_context,
    )

    ablation_models[name] = model
    ablation_grids[name] = grid
    ablation_query_results[name] = test_query_metrics.copy()

    ablation_rows.append({
        "Variant": name,
        "Features": " | ".join(features),
        "Low Threshold": low,
        "High Threshold": high,
        "Target Context": target_context,
        "Validation Fallback Used": fallback,
        "Evidence Precision": test_metrics["precision"],
        "Evidence Recall": test_metrics["recall"],
        "Evidence F1": test_metrics["f1"],
        "Top-1 Relevance": test_metrics["top1"],
        "Query Coverage": test_metrics["coverage"],
        "Abstention Rate": test_metrics["abstention"],
        "Mean Context Count": test_metrics["mean_context"],
    })

ABLATION_SUMMARY = pd.DataFrame(ablation_rows)

ABLATION_SUMMARY["Precision Gain vs Original"] = (
    ABLATION_SUMMARY["Evidence Precision"]
    - ORIGINAL_TEST["precision"]
)

ABLATION_SUMMARY["Recall Change vs Original"] = (
    ABLATION_SUMMARY["Evidence Recall"]
    - ORIGINAL_TEST["recall"]
)

ABLATION_SUMMARY["Context Reduction vs Original (%)"] = (
    1
    - ABLATION_SUMMARY["Mean Context Count"]
    / ORIGINAL_TEST["mean_context"]
) * 100

display(
    ABLATION_SUMMARY[
        [
            "Variant",
            "Low Threshold",
            "High Threshold",
            "Target Context",
            "Evidence Precision",
            "Evidence Recall",
            "Evidence F1",
            "Top-1 Relevance",
            "Query Coverage",
            "Mean Context Count",
            "Precision Gain vs Original",
            "Context Reduction vs Original (%)",
        ]
    ].round(4)
)

ABLATION_SUMMARY.to_csv(
    SAVE_DIR / "Final_Ablation_Summary.csv",
    index=False
)

for name, df in ablation_query_results.items():
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    df.to_csv(
        SAVE_DIR / f"Ablation_Query_Level_{safe_name}.csv",
        index=False
    )


## 5. Full-model coefficient analysis

Because inputs are standardized inside the logistic-regression pipeline, the fitted coefficients provide an interpretable view of the direction and relative magnitude of each confidence signal.

This is **model-level interpretation**, not a causal claim.


In [ ]:
full_model = ablation_models["A0 Full P2"]

coef = (
    full_model
    .named_steps["classifier"]
    .coef_[0]
)

COEFFICIENTS = pd.DataFrame({
    "Feature": ABLATIONS["A0 Full P2"],
    "Standardized Logistic Coefficient": coef,
})

COEFFICIENTS["Absolute Magnitude"] = (
    COEFFICIENTS["Standardized Logistic Coefficient"]
    .abs()
)

COEFFICIENTS = COEFFICIENTS.sort_values(
    "Absolute Magnitude",
    ascending=False
)

display(COEFFICIENTS.round(4))

COEFFICIENTS.to_csv(
    SAVE_DIR / "Full_P2_Confidence_Estimator_Coefficients.csv",
    index=False
)


## 6. Statistical utilities

We use:

- **paired bootstrap 95% CI** over test queries
- **paired sign-flip permutation test**
- **Holm correction** for multiple comparisons

No test-query information is used to tune the model.


In [ ]:
RNG = np.random.default_rng(SEED)

def paired_clean(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    mask = (
        np.isfinite(a)
        & np.isfinite(b)
    )

    return a[mask], b[mask]


def paired_bootstrap_diff(
    a,
    b,
    n_boot=N_BOOT,
    seed=SEED
):
    # Difference = b - a
    a, b = paired_clean(a, b)

    if len(a) == 0:
        return {
            "n": 0,
            "mean_a": np.nan,
            "mean_b": np.nan,
            "mean_diff": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    diff = b - a
    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(diff),
        size=(n_boot, len(diff))
    )

    boot = diff[idx].mean(axis=1)

    return {
        "n": len(diff),
        "mean_a": float(a.mean()),
        "mean_b": float(b.mean()),
        "mean_diff": float(diff.mean()),
        "ci_low": float(np.percentile(boot, 2.5)),
        "ci_high": float(np.percentile(boot, 97.5)),
    }


def paired_sign_flip_pvalue(
    a,
    b,
    n_perm=N_PERM,
    seed=SEED
):
    # H0: paired differences are centered at 0
    a, b = paired_clean(a, b)

    if len(a) == 0:
        return np.nan

    diff = b - a
    observed = abs(diff.mean())

    rng = np.random.default_rng(seed)

    signs = rng.choice(
        [-1.0, 1.0],
        size=(n_perm, len(diff))
    )

    null_means = (
        signs * diff
    ).mean(axis=1)

    p = (
        np.sum(
            np.abs(null_means) >= observed
        ) + 1
    ) / (n_perm + 1)

    return float(p)


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)

    adjusted = np.full(
        len(pvalues),
        np.nan
    )

    valid = np.where(
        np.isfinite(pvalues)
    )[0]

    if len(valid) == 0:
        return adjusted

    order = valid[
        np.argsort(pvalues[valid])
    ]

    m = len(order)
    running_max = 0.0

    for rank, idx in enumerate(order):
        corrected = (
            (m - rank)
            * pvalues[idx]
        )

        running_max = max(
            running_max,
            corrected
        )

        adjusted[idx] = min(
            running_max,
            1.0
        )

    return adjusted


print("Statistical utilities ready.")


## 7. Original Retrieval vs Full P2 — paired test-query statistics

For higher-is-better metrics, positive difference means P2 improved.

For `Context Count`, negative difference means P2 used less evidence.


In [ ]:
FULL_P2_Q = (
    ablation_query_results["A0 Full P2"]
    .copy()
)

paired = ORIGINAL_TEST_Q.merge(
    FULL_P2_Q,
    on="query",
    suffixes=("_original", "_p2")
)

evidence_specs = [
    ("Evidence Precision", "precision_original", "precision_p2", "Higher is better"),
    ("Evidence Recall", "recall_original", "recall_p2", "Higher is better"),
    ("Top-1 Relevance", "top1_original", "top1_p2", "Higher is better"),
    ("Context Count", "context_count_original", "context_count_p2", "Lower is better"),
]

evidence_stat_rows = []

for i, (metric, col_a, col_b, direction) in enumerate(evidence_specs):
    bs = paired_bootstrap_diff(
        paired[col_a],
        paired[col_b],
        seed=SEED + i
    )

    p = paired_sign_flip_pvalue(
        paired[col_a],
        paired[col_b],
        seed=SEED + 100 + i
    )

    evidence_stat_rows.append({
        "Metric": metric,
        "Original Mean": bs["mean_a"],
        "P2 Mean": bs["mean_b"],
        "P2 - Original": bs["mean_diff"],
        "95% CI Low": bs["ci_low"],
        "95% CI High": bs["ci_high"],
        "Permutation p": p,
        "Direction": direction,
        "Paired Queries": bs["n"],
    })

EVIDENCE_STATS = pd.DataFrame(
    evidence_stat_rows
)

EVIDENCE_STATS["Holm-adjusted p"] = holm_adjust(
    EVIDENCE_STATS["Permutation p"].values
)

def ci_interpret(row):
    lo = row["95% CI Low"]
    hi = row["95% CI High"]

    if row["Direction"] == "Higher is better":
        if lo > 0:
            return "P2 improvement supported"
        if hi < 0:
            return "P2 decrease supported"
        return "Difference uncertain"

    else:
        if hi < 0:
            return "P2 reduction supported"
        if lo > 0:
            return "P2 increase supported"
        return "Difference uncertain"

EVIDENCE_STATS["CI Interpretation"] = (
    EVIDENCE_STATS.apply(
        ci_interpret,
        axis=1
    )
)

display(EVIDENCE_STATS.round(4))

EVIDENCE_STATS.to_csv(
    SAVE_DIR / "Final_P2_vs_Original_Evidence_Statistics.csv",
    index=False
)


## 8. Full P2 vs feature ablations

This directly tests whether removing a signal family changes test-query evidence performance.

The table below focuses on **Evidence Precision**, **Evidence Recall**, and **Top-1 Relevance**.


In [ ]:
ablation_stat_rows = []

full_q = ablation_query_results["A0 Full P2"]

for variant, variant_q in ablation_query_results.items():
    if variant == "A0 Full P2":
        continue

    merged = full_q.merge(
        variant_q,
        on="query",
        suffixes=("_full", "_ablated")
    )

    for metric, base_col in [
        ("Evidence Precision", "precision"),
        ("Evidence Recall", "recall"),
        ("Top-1 Relevance", "top1"),
    ]:
        # Difference = Full - Ablated
        bs = paired_bootstrap_diff(
            merged[f"{base_col}_ablated"],
            merged[f"{base_col}_full"],
            seed=SEED + len(ablation_stat_rows)
        )

        p = paired_sign_flip_pvalue(
            merged[f"{base_col}_ablated"],
            merged[f"{base_col}_full"],
            seed=SEED + 500 + len(ablation_stat_rows)
        )

        ablation_stat_rows.append({
            "Ablation": variant,
            "Metric": metric,
            "Ablated Mean": bs["mean_a"],
            "Full P2 Mean": bs["mean_b"],
            "Full - Ablated": bs["mean_diff"],
            "95% CI Low": bs["ci_low"],
            "95% CI High": bs["ci_high"],
            "Permutation p": p,
            "Paired Queries": bs["n"],
        })

ABLATION_STATS = pd.DataFrame(
    ablation_stat_rows
)

ABLATION_STATS["Holm-adjusted p"] = holm_adjust(
    ABLATION_STATS["Permutation p"].values
)

ABLATION_STATS["CI Supports Full > Ablated"] = (
    ABLATION_STATS["95% CI Low"] > 0
)

display(ABLATION_STATS.round(4))

ABLATION_STATS.to_csv(
    SAVE_DIR / "Final_FullP2_vs_Ablations_Statistics.csv",
    index=False
)


## 9. End-to-end Qwen statistical robustness

This section loads the already-generated G1/G2/G3 evaluated file.

The primary final comparison is:

**G1 Original RAG vs G3 P2 Adaptive RAG**

No new LLM generation occurs.


In [ ]:
GEN_EVAL_PATH = (
    SAVE_DIR
    / "G1_G2_G3_Qwen_RAG_Evaluated.csv"
)

if not GEN_EVAL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {GEN_EVAL_PATH}. "
        "Run P2_Adaptive_Confidence_Gate_Full_Colab.ipynb first."
    )

ge = pd.read_csv(GEN_EVAL_PATH)

end_specs = [
    (
        "Context Count",
        "g1_context_count",
        "g3_context_count",
        "Lower is better",
    ),
    (
        "Latency (s)",
        "g1_latency_s",
        "g3_latency_s",
        "Lower is better",
    ),
    (
        "Citation Presence",
        "g1_citation",
        "g3_citation",
        "Higher is better",
    ),
    (
        "Semantic Evidence-Support Proxy",
        "g1_support_proxy",
        "g3_support_proxy",
        "Higher is better",
    ),
    (
        "Dataset-Response Similarity Proxy",
        "g1_response_similarity",
        "g3_response_similarity",
        "Higher is better",
    ),
]

end_stat_rows = []

for i, (metric, g1_col, g3_col, direction) in enumerate(end_specs):
    if g1_col not in ge.columns or g3_col not in ge.columns:
        print("Skipping missing columns:", g1_col, g3_col)
        continue

    bs = paired_bootstrap_diff(
        ge[g1_col],
        ge[g3_col],
        seed=SEED + 1000 + i
    )

    p = paired_sign_flip_pvalue(
        ge[g1_col],
        ge[g3_col],
        seed=SEED + 1500 + i
    )

    end_stat_rows.append({
        "Metric": metric,
        "G1 Original Mean": bs["mean_a"],
        "G3 P2 Mean": bs["mean_b"],
        "G3 - G1": bs["mean_diff"],
        "95% CI Low": bs["ci_low"],
        "95% CI High": bs["ci_high"],
        "Permutation p": p,
        "Direction": direction,
        "Paired Queries": bs["n"],
    })

END_STATS = pd.DataFrame(
    end_stat_rows
)

END_STATS["Holm-adjusted p"] = holm_adjust(
    END_STATS["Permutation p"].values
)

END_STATS["CI Interpretation"] = (
    END_STATS.apply(
        ci_interpret,
        axis=1
    )
)

display(END_STATS.round(4))

END_STATS.to_csv(
    SAVE_DIR / "Final_G1_vs_G3_EndToEnd_Statistics.csv",
    index=False
)


## 10. Paper-ready figures


In [ ]:
# Figure 1 — Ablation evidence performance
plot_cols = [
    "Evidence Precision",
    "Evidence Recall",
    "Evidence F1",
    "Top-1 Relevance",
]

plot_df = (
    ABLATION_SUMMARY
    .set_index("Variant")[plot_cols]
)

ax = plot_df.plot(
    kind="bar",
    figsize=(13, 6)
)

ax.set_ylim(0, 1.02)
ax.set_ylabel("Score")
ax.set_title(
    "Ablation Study — Adaptive Confidence Gate"
)

plt.xticks(
    rotation=25,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    SAVE_DIR / "Figure_Final_Ablation_Study.png",
    dpi=240
)

plt.show()


# Figure 2 — P2 vs Original paired effects with 95% CI
forest = EVIDENCE_STATS.copy()

x = forest["P2 - Original"].values
lo = (
    forest["P2 - Original"]
    - forest["95% CI Low"]
).values

hi = (
    forest["95% CI High"]
    - forest["P2 - Original"]
).values

y = np.arange(len(forest))

fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.errorbar(
    x,
    y,
    xerr=np.vstack([lo, hi]),
    fmt="o",
    capsize=5
)

ax.axvline(
    0,
    linestyle="--"
)

ax.set_yticks(y)
ax.set_yticklabels(
    forest["Metric"]
)

ax.set_xlabel(
    "Paired mean difference (P2 - Original)"
)

ax.set_title(
    "P2 vs Original Retrieval — 95% Bootstrap CI"
)

plt.tight_layout()

plt.savefig(
    SAVE_DIR / "Figure_Final_P2_vs_Original_CI.png",
    dpi=240
)

plt.show()


# Figure 3 — End-to-end G3 vs G1 effects
forest2 = END_STATS.copy()

x = forest2["G3 - G1"].values
lo = (
    forest2["G3 - G1"]
    - forest2["95% CI Low"]
).values

hi = (
    forest2["95% CI High"]
    - forest2["G3 - G1"]
).values

y = np.arange(len(forest2))

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.errorbar(
    x,
    y,
    xerr=np.vstack([lo, hi]),
    fmt="o",
    capsize=5
)

ax.axvline(
    0,
    linestyle="--"
)

ax.set_yticks(y)
ax.set_yticklabels(
    forest2["Metric"]
)

ax.set_xlabel(
    "Paired mean difference (G3 P2 - G1 Original)"
)

ax.set_title(
    "End-to-End P2 vs Original — 95% Bootstrap CI"
)

plt.tight_layout()

plt.savefig(
    SAVE_DIR / "Figure_Final_G3_vs_G1_CI.png",
    dpi=240
)

plt.show()


## 11. Automatic conference-paper statistical summary

The generated text file is intentionally conservative. It reports the measured differences and whether the 95% paired-bootstrap CI excludes zero.

It does not silently turn proxy metrics into accuracy/faithfulness claims.


In [ ]:
def fmt(x):
    if pd.isna(x):
        return "NA"
    return f"{float(x):.4f}"


lines = []

lines.append(
    "FINAL ABLATION & STATISTICAL VALIDATION — AUTOMATIC SUMMARY"
)

lines.append("=" * 78)
lines.append("")

lines.append("EXPERIMENT DESIGN")
lines.append(
    f"Train / Validation / Test queries: "
    f"{train_df['query'].nunique()} / "
    f"{val_df['query'].nunique()} / "
    f"{test_df['query'].nunique()}"
)
lines.append(
    "All confidence thresholds were selected on validation only."
)
lines.append(
    f"Paired bootstrap resamples: {N_BOOT}"
)
lines.append(
    f"Paired sign-flip permutations: {N_PERM}"
)
lines.append("")

lines.append("ABLATION TEST SUMMARY")
lines.append(
    ABLATION_SUMMARY[
        [
            "Variant",
            "Evidence Precision",
            "Evidence Recall",
            "Evidence F1",
            "Top-1 Relevance",
            "Query Coverage",
            "Mean Context Count",
        ]
    ].round(4).to_string(index=False)
)
lines.append("")

lines.append("ORIGINAL RETRIEVAL VS FULL P2 — PAIRED STATISTICS")
lines.append(
    EVIDENCE_STATS[
        [
            "Metric",
            "Original Mean",
            "P2 Mean",
            "P2 - Original",
            "95% CI Low",
            "95% CI High",
            "Holm-adjusted p",
            "CI Interpretation",
        ]
    ].round(4).to_string(index=False)
)
lines.append("")

lines.append("FULL P2 VS ABLATIONS — PAIRED STATISTICS")
lines.append(
    ABLATION_STATS[
        [
            "Ablation",
            "Metric",
            "Ablated Mean",
            "Full P2 Mean",
            "Full - Ablated",
            "95% CI Low",
            "95% CI High",
            "Holm-adjusted p",
        ]
    ].round(4).to_string(index=False)
)
lines.append("")

lines.append("G1 ORIGINAL RAG VS G3 P2 — END-TO-END PAIRED STATISTICS")
lines.append(
    END_STATS[
        [
            "Metric",
            "G1 Original Mean",
            "G3 P2 Mean",
            "G3 - G1",
            "95% CI Low",
            "95% CI High",
            "Holm-adjusted p",
            "CI Interpretation",
        ]
    ].round(4).to_string(index=False)
)

lines.append("")
lines.append("INTERPRETATION RULES")
lines.append(
    "- A 95% CI crossing zero means the direction is not treated as statistically stable."
)
lines.append(
    "- Holm-adjusted p-values are reported for multiple-comparison control."
)
lines.append(
    "- Dataset-response similarity is a semantic proxy, not gold-answer accuracy."
)
lines.append(
    "- Semantic evidence-support is an embedding proxy, not a true faithfulness metric."
)
lines.append(
    "- Logistic-regression coefficients describe model association, not causal importance."
)

summary_text = "\n".join(lines)

SUMMARY_PATH = (
    SAVE_DIR
    / "Final_Ablation_Statistical_Validation_Summary.txt"
)

SUMMARY_PATH.write_text(
    summary_text,
    encoding="utf-8"
)

print(summary_text)
print("\nSaved:", SUMMARY_PATH)


# Finished — Implementation can now be frozen

Main new outputs in `MyDrive/Capstone_Conference_Paper/`:

- `Final_Ablation_Summary.csv`
- `Full_P2_Confidence_Estimator_Coefficients.csv`
- `Final_P2_vs_Original_Evidence_Statistics.csv`
- `Final_FullP2_vs_Ablations_Statistics.csv`
- `Final_G1_vs_G3_EndToEnd_Statistics.csv`
- `Final_Ablation_Statistical_Validation_Summary.txt`
- `Figure_Final_Ablation_Study.png`
- `Figure_Final_P2_vs_Original_CI.png`
- `Figure_Final_G3_vs_G1_CI.png`
- query-level CSVs for every ablation

After this notebook, the recommended next phase is **coding freeze → IEEE conference-paper writing**.
